 # Phase 2: Ingestion & Embeddings
 This interactive script runs the ingestion pipeline to:
 1. Load the generated 200 synthetic invoices from `data/invoices.json`.
 2. Compute 384-dimensional vector embeddings using the `all-MiniLM-L6-v2` SentenceTransformer.
 3. Store the markdown text, raw JSON, and vector arrays inside PostgreSQL.
 4. Verify that data is correctly loaded and indexed.

In [ ]:
# Cell 1: Load Data and Verify DB Connection
import os
import json
import pandas as pd

from src import config
from src.db import get_db_connection
from src.ingest import ingest_invoices

# Setup paths
DATA_DIR = "data"
INVOICES_JSON_PATH = os.path.join(DATA_DIR, "invoices.json")

if not os.path.exists(INVOICES_JSON_PATH):
    raise FileNotFoundError(f"Missing invoice data at '{INVOICES_JSON_PATH}'. Run 01_foundations.py first!")

with open(INVOICES_JSON_PATH, "r") as f:
    invoices = json.load(f)

print(f"Loaded {len(invoices)} invoices from local disk.")

# Quick connection verification
conn = get_db_connection()
conn.close()
print("Database connection verified successfully. Ready for ingestion.")

c:\Users\pnala\Desktop\IDP_Archive\idp_codebase\Digital_Archive_Research\rag_dev\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 200 invoices from local disk.
Database connection verified successfully. Ready for ingestion.


 ## Step 2.2: Execute Ingestion
 We will now pass our dataset through the ingestion processor.
 This cell loads the embedding model locally, vectorizes each invoice's Markdown representation, and upserts it into the `invoice_chunks` table.

In [ ]:
# Cell 2: Run Ingestion
conn = get_db_connection()
try:
    ingest_invoices(invoices, conn, model_name="all-MiniLM-L6-v2")
finally:
    conn.close()

Loading embedding model 'all-MiniLM-L6-v2'...
Embedding model loaded successfully.
Embedded and stored 20/200 invoices...
Embedded and stored 40/200 invoices...
Embedded and stored 60/200 invoices...
Embedded and stored 80/200 invoices...
Embedded and stored 100/200 invoices...
Embedded and stored 120/200 invoices...
Embedded and stored 140/200 invoices...
Embedded and stored 160/200 invoices...
Embedded and stored 180/200 invoices...
Embedded and stored 200/200 invoices...
Successfully ingested 200 invoices into the database.


 ## Step 2.3: Verify Ingested Vector Records
 Let's query the database to verify the stored data and check the dimensions of the saved embeddings.

In [ ]:
# Cell 3: Verify Ingested Records
conn = get_db_connection()
cursor = conn.cursor()

# 1. Get total record count
cursor.execute("SELECT COUNT(*) FROM invoice_chunks;")
count = cursor.fetchone()
record_count = list(count.values())[0] if isinstance(count, dict) else count[0]
print(f"Total records in invoice_chunks: {record_count}")

# 2. Retrieve a sample record to check vector structure
cursor.execute("""
    SELECT invoice_id, content_text, 
           pg_typeof(embedding) as vector_type,
           vector_dims(embedding) as dimensions,
           left(embedding::text, 100) as vector_slice
    FROM invoice_chunks 
    LIMIT 1;
""")
sample = cursor.fetchone()

if sample:
    print("\n--- Ingested Vector Record Sample ---")
    for key, val in sample.items() if isinstance(sample, dict) else enumerate(sample):
        print(f"{key}: {val}")
else:
    print("No records found. Ingestion may have failed.")

conn.close()

Total records in invoice_chunks: 200

--- Ingested Vector Record Sample ---
invoice_id: INV-2025-0001
content_text: # Invoice INV-2025-0001

- **Invoice Id**: INV-2025-0001
- **Vendor**: Summit Hardware
- **Date**: 2026-03-21
- **Buyer**: Johnson LLC
- **Buyer Address**: 181 Johnson Course, East William, AK 74064
## Line Items
1. **Description**: Laptop | **Qty**: 12 | **Unit Price**: 1267.55 | **Amount**: 15210.6

- **Subtotal**: 15210.6
- **Tax Rate**: 6%
- **Tax**: 912.64
- **Grand Total**: 16123.24
- **Currency**: MYR
- **Payment Terms**: Due on Receipt
vector_type: vector
dimensions: 384
vector_slice: [-0.069530584,0.076768205,0.0125073,-0.031956434,-0.041033875,-0.016661473,-0.059921622,0.007241367,
